# Imports

In [ ]:
import json
import os
import platform
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort
import pandas as pd
import torch
from tqdm.auto import tqdm

from uavloc.data.drone_frames import DroneFrame
from uavloc.data.satellite import GeoImage
from uavloc.features.segmentation import SegmentationFeatures
from uavloc.geo import LatLon, haversine_m
from uavloc.matching.correlator import Correction, CorrelatorConfig, PositionCorrelator, decide

# Params

In [ ]:
MODEL_DIR = "models"
CASE_DIR = "case"
LOG_ROOT = "runs/logs"
RUN_ID = ""
ARCHITECTURES = ["unet-resnet18", "unet-resnet34", "lraspp-mobilenetv3"]
THREADS = 4
N_WARMUP = 5
N_ITERS = 30
COOLDOWN_S = 10
PARITY_TOL_M = 0.05                              # Pi vs Mac corrected position
MS_PER_S = 1000.0

TEMP_PATH = "/sys/class/thermal/thermal_zone0/temp"
FREQ_PATH = "/sys/devices/system/cpu/cpu0/cpufreq/scaling_cur_freq"
GOVERNOR_PATH = "/sys/devices/system/cpu/cpu0/cpufreq/scaling_governor"
CPUINFO_PATH = "/proc/cpuinfo"
VCGENCMD = "vcgencmd"

# Data

In [ ]:
case = json.loads((Path(CASE_DIR) / "case.json").read_text())
frame = DroneFrame(case["frame"]["name"], Path(CASE_DIR) / "frame.jpg", LatLon(*case["frame"]["position"]),
                   case["frame"]["altitude_m"], case["frame"]["yaw_deg"], case["frame"]["mpp"], ())
prior = LatLon(*case["prior"])
config = CorrelatorConfig(**{**case["correlator_config"],
                             "class_weights": tuple(case["correlator_config"]["class_weights"])})
frame.name, frame.image().size, round(haversine_m(prior, frame.position), 2), config

In [ ]:
def load_probabilities(arch: str) -> GeoImage:
    """The satellite class-probability window segmented before flight (on the Mac) by `arch`."""
    data = np.load(Path(CASE_DIR) / f"satellite-{arch}.npz")
    return GeoImage(data["pixels"], LatLon(*data["top_left"]), LatLon(*data["bottom_right"]))

# Model

In [ ]:
def sensors() -> dict:
    throttled = subprocess.run([VCGENCMD, "get_throttled"], capture_output=True, text=True, check=True).stdout
    return {"temp_c": int(Path(TEMP_PATH).read_text()) / 1000,
            "cpu_freq_mhz": int(Path(FREQ_PATH).read_text()) / 1000,
            "throttled": throttled.strip().split("=")[-1]}

In [ ]:
def system_info() -> dict:
    cpuinfo = Path(CPUINFO_PATH).read_text().splitlines()
    board = next(line.split(":", 1)[1].strip() for line in cpuinfo if line.startswith("Model"))
    return {"board": board, "platform": platform.platform(), "python": platform.python_version(),
            "torch": torch.__version__, "onnxruntime": ort.__version__, "numpy": np.__version__,
            "cpu_count": os.cpu_count(), "governor": Path(GOVERNOR_PATH).read_text().strip(),
            "time_utc": datetime.now(timezone.utc).isoformat(), **sensors()}

In [ ]:
class OrtSegmenter(torch.nn.Module):
    """An onnxruntime session behind the `model(batch)` call of `class_probabilities`; keeps each run's time."""

    def __init__(self, path: Path, threads: int) -> None:
        super().__init__()
        options = ort.SessionOptions()
        options.intra_op_num_threads = threads
        options.inter_op_num_threads = 1
        self.session = ort.InferenceSession(str(path), options, providers=["CPUExecutionProvider"])
        self.inference_ms: list[float] = []

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        start = time.perf_counter()
        logits = self.session.run(None, {"image": x.numpy()})[0]
        self.inference_ms.append((time.perf_counter() - start) * MS_PER_S)
        return torch.from_numpy(logits)

In [ ]:
def build_correlator(arch: str) -> tuple[PositionCorrelator, OrtSegmenter]:
    segmenter = OrtSegmenter(Path(MODEL_DIR) / f"{arch}.onnx", THREADS)
    features = SegmentationFeatures.from_satellite_probabilities(
        segmenter, load_probabilities(arch), case["seg_channels"], case["drone_seg_gsd_m"], case["match_gsd_m"], "cpu")
    return PositionCorrelator(features, config), segmenter

In [ ]:
def timed_correction(correlator: PositionCorrelator, segmenter: OrtSegmenter) -> tuple[Correction, dict]:
    """One correction from a cold drone cache, in ms per stage.

    drone_map = read + resize + segment + softmax; match = cut the satellite patch + weighted NCC surface;
    decide = posterior + gate + move.
    """
    correlator.clear_cache()
    t0 = time.perf_counter()
    correlator.drone_map(frame)
    t1 = time.perf_counter()
    surface = correlator.surface(frame, prior)
    t2 = time.perf_counter()
    correction = decide(surface, correlator.config)
    t3 = time.perf_counter()
    stages = {"drone_map_ms": t1 - t0, "match_ms": t2 - t1, "decide_ms": t3 - t2, "total_ms": t3 - t0}
    return correction, {"inference_ms": segmenter.inference_ms[-1], **{k: v * MS_PER_S for k, v in stages.items()}}

In [ ]:
def run_model(arch: str) -> tuple[Correction, list[dict], float]:
    """N_ITERS timed corrections of the case with `arch`, the last correction, and the model load time."""
    start = time.perf_counter()
    correlator, segmenter = build_correlator(arch)
    load_ms = (time.perf_counter() - start) * MS_PER_S
    for _ in range(N_WARMUP):
        timed_correction(correlator, segmenter)
    rows = []
    bar = tqdm(range(N_ITERS), desc=arch)
    for i in bar:
        correction, row = timed_correction(correlator, segmenter)
        rows.append({"architecture": arch, "iter": i, **row, **sensors()})
        bar.set_postfix(total_ms=f"{row['total_ms']:.0f}")
    return correction, rows, load_ms

In [ ]:
def result_row(arch: str, correction: Correction) -> dict:
    """Error before / after, and how far the Pi's corrected position is from the Mac's."""
    reference = case["references"][arch]
    pi_vs_mac_m = haversine_m(correction.position, LatLon(reference["corrected_lat"], reference["corrected_lon"]))
    return {"architecture": arch, "prior_error_m": haversine_m(prior, frame.position),
            "error_m": haversine_m(correction.position, frame.position), "mac_error_m": reference["error_m"],
            "decision": correction.decision, "peak_z": correction.peak_z, "pi_vs_mac_m": pi_vs_mac_m,
            "matches_mac": pi_vs_mac_m <= PARITY_TOL_M}

# Run

In [ ]:
log_dir = Path(LOG_ROOT) / (RUN_ID or datetime.now().strftime("%Y%m%d-%H%M%S"))
log_dir.mkdir(parents=True)
system_start = system_info()
log_dir, system_start

In [ ]:
corrections, timings, load_ms = {}, [], {}
for arch in ARCHITECTURES:
    corrections[arch], rows, load_ms[arch] = run_model(arch)
    timings += rows
    time.sleep(COOLDOWN_S)

In [ ]:
timings = pd.DataFrame(timings)
timings.to_csv(log_dir / "timings.csv", index=False)
results = pd.DataFrame([result_row(arch, corrections[arch]) for arch in ARCHITECTURES]).set_index("architecture")
results.round(3)

In [ ]:
stages = ["inference_ms", "drone_map_ms", "match_ms", "decide_ms", "total_ms"]
by_arch = timings.groupby("architecture")
summary = (by_arch[stages].median()
           .join(by_arch["total_ms"].quantile(0.9).rename("p90_total_ms"))
           .join(pd.Series(load_ms, name="load_ms"))
           .join(by_arch["temp_c"].max().rename("max_temp_c"))
           .loc[ARCHITECTURES])
summary.round(1)

In [ ]:
(log_dir / "run.json").write_text(json.dumps(
    {"case": {k: case[k] for k in ("experiment", "loss", "rule", "place", "noise_m", "bearing_deg", "prior", "frame")},
     "threads": THREADS, "n_iters": N_ITERS, "n_warmup": N_WARMUP, "system_start": system_start,
     "system_end": system_info(), "results": results.reset_index().to_dict("records"),
     "summary_median_ms": summary.reset_index().to_dict("records")}, indent=2, default=str))

# Plots

In [ ]:
def plot_stages(summary: pd.DataFrame) -> plt.Figure:
    """Median ms per stage of one correction; `drone_map` is split into ONNX inference and the rest."""
    parts = pd.DataFrame({"segment (ONNX)": summary["inference_ms"],
                          "read + resize + softmax": summary["drone_map_ms"] - summary["inference_ms"],
                          "satellite patch + NCC": summary["match_ms"], "decide": summary["decide_ms"]})
    ax = parts.plot.barh(stacked=True, figsize=(9, 0.6 * len(parts) + 1.5))
    ax.set_xlabel(f"ms, median of {N_ITERS}, {THREADS} threads")
    ax.invert_yaxis()
    return ax.figure

In [ ]:
fig = plot_stages(summary)
fig.savefig(log_dir / "stages.png", bbox_inches="tight")

In [ ]:
def plot_heatmaps(corrections: dict[str, Correction]) -> plt.Figure:
    """Each model's search surface (right/down metres from the prior), the truth and the applied move."""
    truth = PositionCorrelator.patch_offset_m(prior, frame.position, frame.yaw_deg)
    fig, axes = plt.subplots(1, len(corrections), figsize=(5 * len(corrections), 4), squeeze=False)
    for ax, (arch, correction) in zip(axes[0], corrections.items()):
        ax.imshow(correction.heatmap, extent=correction.heatmap_extent_m, cmap="magma")
        ax.plot(*truth, "c+", ms=14, mew=2, label="truth")
        ax.plot(*correction.offset_m, "wx", ms=10, mew=2, label="corrected")
        ax.plot(0, 0, "o", mfc="none", mec="w", ms=8, label="prior")
        ax.set_title(f"{arch}: {results.loc[arch, 'prior_error_m']:.1f} -> {results.loc[arch, 'error_m']:.2f} m, "
                     f"{summary.loc[arch, 'total_ms']:.0f} ms")
        ax.set_xlabel("right, m")
    axes[0, 0].set_ylabel("down, m")
    axes[0, 0].legend(loc="lower left", fontsize=8)
    fig.tight_layout()
    return fig

In [ ]:
fig = plot_heatmaps(corrections)
fig.savefig(log_dir / "heatmaps.png", bbox_inches="tight")